In [1]:
#2d implementation progression

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import time

torch.manual_seed(123)
np.random.seed(123)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


In [ ]:
from pinn_shared import (
    set_seed, pinn_architecture, sample_points, generate_noisy_data,
    compute_loss, train_forward, compute_loss_inverse, train_inverse,
    fd_solver, cn_nls_baseline,
)

In [ ]:
forward_config = {
    "N_f": 10000,
    "N_bc": 200,
    "N_ic": 200,
    "adam_lr": 1e-3,
    "adam_iters": 2000,
    "lbfgs_iters": 500,
    "hidden_size": 20,
    "n_layers": 4,
    "true_alpha": 0.4,
}

In [ ]:
# Held-out test grid, shared by both the baseline and the tuned-config
# retrain loop below -- built once, here, so neither section has to come
# first to "own" the grid the other one also needs.
x_test = torch.linspace(0, 1, 1000)
t_test = torch.linspace(0, 1, 1000)
X_test, T_test = torch.meshgrid(x_test, t_test, indexing='ij')
x_test_flat = X_test.reshape(-1, 1)
t_test_flat = T_test.reshape(-1, 1)
u_exact_test = torch.sin(torch.pi * x_test_flat) * torch.exp(-forward_config["true_alpha"] * (torch.pi**2) * t_test_flat)

In [ ]:
# Baseline (no-tuning) forward PINN: the literature-informed hyperparameters
# from heat_pinn_basic.ipynb (forward_config above), run through the same
# multi-seed / held-out-test-grid methodology as the tuned-config retrain
# loop below -- this is the "PINN performance you'd get without paying for
# Optuna tuning" number.
baseline_forward_results = []
for seed in range(5):
    set_seed(seed)
    start = time.time()
    results = train_forward(forward_config, print_training=False)
    results['elapsed'] = time.time() - start
    baseline_forward_results.append(results)

baseline_val_errors = [r['rel_l2'] for r in baseline_forward_results]
baseline_val_times = [r['elapsed'] for r in baseline_forward_results]
print(f"\nBaseline (no tuning):")
print(f"  Error (validation grid): {np.mean(baseline_val_errors):.6e} ± {np.std(baseline_val_errors):.6e}")
print(f"  Time:  {np.mean(baseline_val_times):.2f} ± {np.std(baseline_val_times):.2f} sec")

# Evaluate every seed on the shared held-out TEST grid built above, so
# baseline vs. tuned vs. CN are all reported on identical points. The test
# grid is CPU by default -- moved to whichever device each model actually
# trained on (GPU by default, or CPU if that run explicitly requested the
# CPU control-timing path), then brought back to CPU immediately after.
# Field-evaluation time is timed per seed too -- CLAUDE.md requires PINN
# field-evaluation time as its own reported cost line, and this used to be
# measured for the tuned model only (pinn_infer_time below) while baseline's
# happened inside this same loop without ever being timed.
baseline_test_errors = []
baseline_infer_times = []
for r in baseline_forward_results:
    model_device = next(r["model"].parameters()).device
    infer_start = time.time()
    with torch.no_grad():
        u_pred_test = r["model"](x_test_flat.to(model_device), t_test_flat.to(model_device)).cpu()
    baseline_infer_times.append(time.time() - infer_start)
    baseline_test_errors.append((torch.norm(u_pred_test - u_exact_test) / torch.norm(u_exact_test)).item())

baseline_test_rel_l2_mean = float(np.mean(baseline_test_errors))
baseline_test_rel_l2_std = float(np.std(baseline_test_errors))
baseline_train_time_mean = float(np.mean(baseline_val_times))
baseline_train_time_std = float(np.std(baseline_val_times))
baseline_infer_time_mean = float(np.mean(baseline_infer_times))
baseline_infer_time_std = float(np.std(baseline_infer_times))

print(f"  Held-out test rel L2 error: {baseline_test_rel_l2_mean:.6e} ± {baseline_test_rel_l2_std:.6e}")
print(f"  Training time:              {baseline_train_time_mean:.2f} ± {baseline_train_time_std:.2f} sec")
print(f"  Field-evaluation time:      {baseline_infer_time_mean:.4f} ± {baseline_infer_time_std:.4f} sec")

In [ ]:
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)

# Everything about the forward problem that Optuna does NOT search over --
# defined once here so both objective_forward (HPO trials) and the
# top-configs retrain loop later (9c4dc518) build configs from the same
# source instead of two copies that could drift out of sync. true_alpha
# matches forward_config["true_alpha"] above -- both are the same fixed
# physical constant of the experiment (mirrors how INVERSE_FIXED_CONFIG
# and INVERSE_OBS_CONFIG both carry inverse's true_alpha).
FORWARD_FIXED_CONFIG = {
    "lambda_pde": 1,
    "true_alpha": 0.4,
}

def objective_forward(trial):
    config = {
        **FORWARD_FIXED_CONFIG,
        "hidden_size":  trial.suggest_categorical("hidden_size", [16, 32, 64, 128]),
        "n_layers":     trial.suggest_categorical("n_layers", [3, 4, 5]),
        "activation":   trial.suggest_categorical("activation", ["tanh", "sin"]),
        "lambda_bc":    trial.suggest_float("lambda_bc",  0.001, 1000.0, log=True),
        "lambda_ic":    trial.suggest_float("lambda_ic",  0.001, 1000.0, log=True),
        "N_f":          trial.suggest_categorical("N_f", [5000, 10000, 20000]),
        "N_bc":         trial.suggest_categorical("N_bc", [100, 200, 400]),
        "N_ic":         trial.suggest_categorical("N_ic", [100, 200, 400]),
        "adam_lr":      trial.suggest_float("adam_lr", 1e-4, 1e-2, log=True),
        "adam_iters":   1500,
        "lbfgs_iters":  500,
    }

    results = train_forward(config, print_training = False, trial = trial)
    return results["rel_l2"]

In [7]:
import os
if os.path.exists("optuna_forward.db"):
    os.remove("optuna_forward.db")


In [ ]:
import joblib
import optuna.visualization as vis

study = optuna.create_study(
    direction="minimize",
    study_name="forward_pinn_hpo",
    storage="sqlite:///optuna_forward.db",
    load_if_exists=True,
    pruner=optuna.pruners.MedianPruner(n_startup_trials=15, n_warmup_steps=800, interval_steps=200),
    sampler=optuna.samplers.TPESampler(seed=42)
)

optuna_forward_search_start = time.time()
study.optimize(
    objective_forward, 
    n_trials=300,
    show_progress_bar=True
)
optuna_forward_search_time = time.time() - optuna_forward_search_start

joblib.dump(study, 'forward_hpo_study.pkl')

print(f"\nTotal Optuna forward search time: {optuna_forward_search_time:.2f}s")
print("\nBest trial:")
print(f"  rel_l2: {study.best_trial.value:.4e}")
print(f"  params: {study.best_trial.params}")

vis.plot_optimization_history(study).show()
vis.plot_param_importances(study).show()

In [9]:
import optuna

study = optuna.load_study(
    study_name="forward_pinn_hpo",
    storage="sqlite:///optuna_forward.db"
)

print(f"\nBest trial:")
print(f"  Error: {study.best_value:.6e}")
print(f"  Config: {study.best_params}")

# See all top configs
print(f"\nTop 5 trials:")
sorted_trials = sorted(study.trials, key=lambda t: t.value if t.value is not None else float('inf'))
for i, trial in enumerate(sorted_trials[:5], 1):
    print(f"{i}. Error: {trial.value:.6e}")
    print(f"   Params: {trial.params}\n")

# Stats
completed = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
pruned = [t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED]
print(f"Summary:")
print(f"  Completed: {len(completed)}")
print(f"  Pruned: {len(pruned)}")
print(f"  Pruning saved: {len(pruned)/(len(completed)+len(pruned))*100:.1f}% of trials")


Best trial:
  Error: 1.692019e-04
  Config: {'hidden_size': 128, 'n_layers': 4, 'activation': 'tanh', 'lambda_bc': 16.64287157253686, 'lambda_ic': 1.3287678633804623, 'N_f': 10000, 'N_bc': 200, 'N_ic': 400, 'adam_lr': 0.005104288463439231}

Top 5 trials:
1. Error: 1.692019e-04
   Params: {'hidden_size': 128, 'n_layers': 4, 'activation': 'tanh', 'lambda_bc': 16.64287157253686, 'lambda_ic': 1.3287678633804623, 'N_f': 10000, 'N_bc': 200, 'N_ic': 400, 'adam_lr': 0.005104288463439231}

2. Error: 1.781247e-04
   Params: {'hidden_size': 128, 'n_layers': 5, 'activation': 'tanh', 'lambda_bc': 18.312606180932846, 'lambda_ic': 1.2172733015396069, 'N_f': 10000, 'N_bc': 200, 'N_ic': 400, 'adam_lr': 0.004818229052622651}

3. Error: 1.845841e-04
   Params: {'hidden_size': 128, 'n_layers': 4, 'activation': 'tanh', 'lambda_bc': 7.342508201207911, 'lambda_ic': 1.7281707586350386, 'N_f': 10000, 'N_bc': 200, 'N_ic': 400, 'adam_lr': 0.00753569599541483}

4. Error: 2.000675e-04
   Params: {'hidden_size': 1

In [ ]:
# Retrain candidates pulled directly from the completed Optuna study (top 3
# by validation error), not hand-transcribed numbers -- the notebook runs
# end-to-end without anyone reading printed trial output and typing values
# back in. FORWARD_FIXED_CONFIG supplies what Optuna didn't search over;
# adam_iters/lbfgs_iters are bumped from the HPO trial's shorter budget
# (1500/500) for a more thorough final retrain.
top_configs = [
    {**FORWARD_FIXED_CONFIG, **trial.params, "adam_iters": 3000, "lbfgs_iters": 1500}
    for trial in sorted_trials[:3]
]

# Train every (config, seed) pair and keep every result -- not just the
# single luckiest one -- so config selection and final reporting can both
# be based on average behavior instead of a cherry-picked best run.
all_config_results = []

for i, config in enumerate(top_configs, 1):
    config_results = []
    for seed in range(5):
        set_seed(seed)
        start = time.time()
        results = train_forward(config, print_training=False)
        results['elapsed'] = time.time() - start
        config_results.append(results)

    errors = [r['rel_l2'] for r in config_results]
    times = [r['elapsed'] for r in config_results]
    print(f"\nConfig {i}:")
    print(f"  Error (validation grid): {np.mean(errors):.6e} ± {np.std(errors):.6e}")
    print(f"  Time:  {np.mean(times):.2f} ± {np.std(times):.2f} sec")

    all_config_results.append(config_results)

# Pick the winning config by its MEAN validation error across 5 seeds, not
# by any single individual run's error -- so config selection isn't biased
# toward whichever run happened to get an easy random draw.
config_mean_errors = [np.mean([r['rel_l2'] for r in cr]) for cr in all_config_results]
best_config_idx = int(np.argmin(config_mean_errors))
winning_results = all_config_results[best_config_idx]

# Evaluate every seed of the winning config on the held-out TEST grid
# (never used during HPO or config selection) -- this mean/std, not any
# single run's number, is what should be quoted as "PINN accuracy." Test
# grid is moved to whichever device each model actually trained on (GPU by
# default, or CPU for an explicit CPU control-timing run), then brought
# back to CPU right after.
test_errors = []
for r in winning_results:
    model_device = next(r["model"].parameters()).device
    with torch.no_grad():
        u_pred_test = r["model"](x_test_flat.to(model_device), t_test_flat.to(model_device)).cpu()
    test_errors.append((torch.norm(u_pred_test - u_exact_test) / torch.norm(u_exact_test)).item())

pinn_test_rel_l2_mean = float(np.mean(test_errors))
pinn_test_rel_l2_std = float(np.std(test_errors))
pinn_train_time_mean = float(np.mean([r["elapsed"] for r in winning_results]))
pinn_train_time_std = float(np.std([r["elapsed"] for r in winning_results]))

print(f"\nWinning config: Config {best_config_idx + 1}")
print(f"  Held-out test rel L2 error: {pinn_test_rel_l2_mean:.6e} ± {pinn_test_rel_l2_std:.6e}")
print(f"  Training time:              {pinn_train_time_mean:.2f} ± {pinn_train_time_std:.2f} sec")

# Use whichever seed's validation error is closest to the winning config's
# mean as a single representative model for the plots below -- illustrative
# only. The mean ± std above are the numbers that should be quoted as PINN
# performance, not this one run's individual numbers.
representative_idx = int(np.argmin([abs(r['rel_l2'] - config_mean_errors[best_config_idx]) for r in winning_results]))
best_result = winning_results[representative_idx]

model = best_result["model"]
pinn_train_time = best_result["train_time"]

In [ ]:
# Reuse the held-out test grid built above (x_test/t_test/X_test/T_test/...)
# instead of rebuilding an identical one from scratch -- same points, so
# there's no reason to maintain two copies of the same grid construction.
x, t = x_test, t_test
X, T = X_test, T_test
x_flat, t_flat = x_test_flat, t_test_flat

# Move inputs to whichever device the winning model actually trained on
# (GPU by default, or CPU for an explicit CPU control-timing run), and
# bring the prediction back to CPU right after -- everything downstream
# (numpy conversion, plotting, error math) is unchanged.
model_device = next(model.parameters()).device
pinn_infer_start = time.time()
with torch.no_grad():
    u_pred = model(x_flat.to(model_device), t_flat.to(model_device)).cpu()
pinn_infer_time = time.time() - pinn_infer_start

u_exact = u_exact_test

error = torch.norm(u_pred - u_exact) / torch.norm(u_exact)
linf_error_pinn = torch.max(torch.abs(u_pred - u_exact)).item()

print(f"PINN inference time : {pinn_infer_time:.4f}s")
print(f"PINN rel L2 error   : {error.item():.4e}")
print(f"PINN L-inf error    : {linf_error_pinn:.4e}")

u_pred_grid = u_pred.reshape(1000, 1000)
u_exact_grid = u_exact.reshape(1000, 1000)
u_abs_error = torch.abs(u_pred - u_exact).reshape(1000, 1000)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Plot 1 - exact solution
im1 = axes[0].pcolormesh(T.numpy(), X.numpy(), u_exact_grid.numpy(), cmap='hot')
axes[0].set_title('Exact Solution')
axes[0].set_xlabel('t')
axes[0].set_ylabel('x')
plt.colorbar(im1, ax=axes[0])

# Plot 2 - PINN prediction
im2 = axes[1].pcolormesh(T.numpy(), X.numpy(), u_pred_grid.numpy(), cmap='hot')
axes[1].set_title('PINN Prediction')
axes[1].set_xlabel('t')
axes[1].set_ylabel('x')
plt.colorbar(im2, ax=axes[1])

# Plot 3 - absolute error
im3 = axes[2].pcolormesh(T.numpy(), X.numpy(), u_abs_error.numpy(), cmap='hot')
axes[2].set_title('Absolute Error')
axes[2].set_xlabel('t')
axes[2].set_ylabel('x')
plt.colorbar(im3, ax=axes[2])

plt.tight_layout()
plt.show()

In [ ]:
conv_errors = []
N_values = [50, 100, 200, 400, 800]

for nx in N_values:
    x_c, t_c, u_c, conv_error = fd_solver(nx, nx * 2)
    conv_errors.append(conv_error)

In [ ]:
plt.figure()
plt.loglog(N_values, conv_errors, 'bo-', label='CN Error')
plt.xlabel('N_x')
plt.ylabel('Relative L2 Error')
plt.title('FD Convergence')
plt.grid(True)
plt.show()

In [ ]:
fd_start = time.time()
x_fd, t_fd, u_fd, _ = fd_solver(1000, 1000, compute_error=False)
fd_time = time.time() - fd_start

X_fd, T_fd = np.meshgrid(x_fd, t_fd, indexing='ij')
u_exact_fd = np.sin(np.pi * X_fd) * np.exp(-forward_config["true_alpha"] * (np.pi**2) * T_fd)
error_fd = np.linalg.norm(u_fd - u_exact_fd) / np.linalg.norm(u_exact_fd)
linf_error_fd = np.max(np.abs(u_fd - u_exact_fd))

fig, axes = plt.subplots(1, 5, figsize=(20, 4))

# Exact
im1 = axes[0].pcolormesh(T.numpy(), X.numpy(), u_exact_grid.numpy(), cmap='hot')
axes[0].set_title('Exact')
axes[0].set_xlabel('t')
axes[0].set_ylabel('x')
plt.colorbar(im1, ax=axes[0])

# PINN prediction
im2 = axes[1].pcolormesh(T.numpy(), X.numpy(), u_pred_grid.numpy(), cmap='hot')
axes[1].set_title('PINN')
axes[1].set_xlabel('t')
plt.colorbar(im2, ax=axes[1])

# FD prediction
im3 = axes[2].pcolormesh(T.numpy(), X.numpy(), u_fd, cmap='hot')
axes[2].set_title('FD (CN)')
axes[2].set_xlabel('t')
plt.colorbar(im3, ax=axes[2])

# PINN error
im4 = axes[3].pcolormesh(T.numpy(), X.numpy(), u_abs_error.numpy(), cmap='hot')
axes[3].set_title('PINN Error')
axes[3].set_xlabel('t')
plt.colorbar(im4, ax=axes[3])

# FD error
im5 = axes[4].pcolormesh(T.numpy(), X.numpy(), np.abs(u_fd - u_exact_fd), cmap='hot')
axes[4].set_title('FD Error')
axes[4].set_xlabel('t')
plt.colorbar(im5, ax=axes[4])

plt.tight_layout()
plt.show()

# Note: the "Tuned" column's non-indented rows are a single representative
# run (the seed closest to the winning config's mean), shown for
# illustration alongside the plots above. The mean ± std rows -- for both
# Baseline and Tuned -- are what should be quoted as PINN performance, not
# any single run's individual numbers. Baseline's inference-time row is
# mean ± std only (no single-representative-run value) since baseline never
# singles out one representative model the way the tuned retrain loop does
# with best_result -- every baseline number is already a 5-seed aggregate.
print(f"{'':25} {'Baseline':>14} {'Tuned':>14} {'FD (CN)':>14}")
print(f"{'-'*70}")
print(f"{'Training time (s)':25} {'N/A':>14} {pinn_train_time:>14.2f} {'N/A':>14}")
print(f"{'  mean ± std (5 seeds)':25} {f'{baseline_train_time_mean:.2f}±{baseline_train_time_std:.2f}':>14} {f'{pinn_train_time_mean:.2f}±{pinn_train_time_std:.2f}':>14} {'N/A':>14}")
print(f"{'Inference time (s)':25} {'N/A':>14} {pinn_infer_time:>14.4f} {fd_time:>14.4f}")
print(f"{'  mean ± std (5 seeds)':25} {f'{baseline_infer_time_mean:.4f}±{baseline_infer_time_std:.4f}':>14} {'N/A':>14} {'N/A':>14}")
print(f"{'Rel L2 error':25} {'N/A':>14} {error.item():>14.4e} {error_fd:>14.4e}")
print(f"{'  mean ± std (5 seeds)':25} {f'{baseline_test_rel_l2_mean:.4e}±{baseline_test_rel_l2_std:.4e}':>14} {f'{pinn_test_rel_l2_mean:.4e}±{pinn_test_rel_l2_std:.4e}':>14} {'N/A':>14}")
print(f"{'L-inf error':25} {'N/A':>14} {linf_error_pinn:>14.4e} {linf_error_fd:>14.4e}")
print(f"{'Optuna search time (s)':25} {'N/A':>14} {optuna_forward_search_time:>14.2f} {'N/A':>14}")

In [ ]:
# Baseline (no-tuning) inverse PINN: the exact hyperparameters originally
# used in heat_pinn_basic.ipynb's inverse cell (hardcoded tanh activation,
# all loss terms unweighted), the same way forward_config captures the
# original forward hyperparameters.
baseline_inverse_config = {
    "activation":   "tanh",
    "hidden_size":  20,
    "n_layers":     4,
    "lambda_pde":   1.0,
    "lambda_bc":    1.0,
    "lambda_ic":    1.0,
    "lambda_data":  1.0,
    "N_f":          10000,
    "N_bc":         200,
    "N_ic":         200,
    "N_obs":        10,
    "noise_std":    0.05,
    "alpha_init":   0.1,
    "true_alpha":   0.4,
    "adam_lr":      1e-3,
    "adam_iters":   5000,
    "lbfgs_iters":  500,
}

In [ ]:
# Generate the baseline's own observations once, kept fixed across all 5
# seeds below -- same reasoning as the shared x_obs_inv/t_obs_inv/u_obs_inv
# used for Optuna trials further down: comparing seeds should only reflect
# training stochasticity, not a fresh random observation draw each time.
set_seed(0)
x_obs_baseline, t_obs_baseline, u_obs_baseline = generate_noisy_data(baseline_inverse_config)

baseline_inverse_results = []
for seed in range(5):
    set_seed(seed)
    r = train_inverse(baseline_inverse_config, x_obs_baseline, t_obs_baseline, u_obs_baseline, print_training=False)
    baseline_inverse_results.append(r)

baseline_alpha_errors = [r["alpha_error"] for r in baseline_inverse_results]
baseline_inverse_times = [r["train_time"] for r in baseline_inverse_results]

print(f"\nBaseline (no tuning) inverse PINN:")
print(f"  Alpha error: {np.mean(baseline_alpha_errors):.6e} ± {np.std(baseline_alpha_errors):.6e}")
print(f"  Time: {np.mean(baseline_inverse_times):.2f} ± {np.std(baseline_inverse_times):.2f} sec")

In [ ]:
# Parameters for generating the inverse problem's synthetic noisy
# observations -- named once here (rather than as a bare literal) since
# forthcoming sensitivity sweeps will vary N_obs and noise_std, and every
# generate_noisy_data(...) call in this section (search stage here, plus
# the confirmation-round and final-evaluation cells below) should draw
# from the same definition of the observation setup rather than each
# retyping its own copy of the same three numbers.
INVERSE_OBS_CONFIG = {
    "N_obs": 50,
    "noise_std": 0.05,
    "true_alpha": 0.4,
}

# Generate the inverse-problem observations once, shared across every Optuna
# trial, so trials are compared on the identical noisy dataset rather than
# each drawing its own random noise realization.
set_seed(0)
x_obs_inv, t_obs_inv, u_obs_inv = generate_noisy_data(INVERSE_OBS_CONFIG)

# Everything about the inverse problem that Optuna does NOT search over --
# defined once here so both objective_inverse (HPO trials) and the
# multi-seed retrain loop later (after be0839c3) build configs from the
# same source instead of two copies that could drift out of sync. N_obs
# and noise_std are deliberately not here: train_inverse never reads them
# (observations are passed in pre-built above), so including them would
# just be dead weight that looks like it configures something it doesn't.
#
# activation, N_f, and the N_bc/N_ic range now match forward's search space
# exactly (previously activation was fixed to "sin" and N_f fixed to 10000,
# with N_bc/N_ic searched over double forward's range) -- decided August 21,
# 2026: the original narrower inverse space was sized for MacBook CPU
# training time, not for any physics-based reason the two problems should
# search differently. Forward's own completed HPO run found tanh
# outperforming sin on a closely related problem, which undercuts the
# "sin is the right basis so fix it" assumption inverse was built on.
# Whether inverse specifically benefits from denser BC/IC point coverage
# (a real, separate hypothesis -- more boundary/IC anchoring could help
# alpha identifiability given only sparse noisy data) is deferred to a
# dedicated follow-up ablation rather than left as an untested asymmetry
# baked into the main comparison.
INVERSE_FIXED_CONFIG = {
    "lambda_pde":  1.0,
    "alpha_init":  0.1,
    "true_alpha":  0.4,
}

def objective_inverse(trial):
    config = {
        **INVERSE_FIXED_CONFIG,
        "hidden_size":  trial.suggest_categorical("hidden_size", [16, 32, 64, 128]),
        "n_layers":     trial.suggest_categorical("n_layers", [3, 4, 5]),
        "activation":   trial.suggest_categorical("activation", ["tanh", "sin"]),
        "lambda_bc":    trial.suggest_float("lambda_bc",  0.001, 1000.0, log=True),
        "lambda_ic":    trial.suggest_float("lambda_ic",  0.001, 1000.0, log=True),
        "lambda_data":  trial.suggest_float("lambda_data",  0.001, 1000.0, log=True),
        "N_f":          trial.suggest_categorical("N_f", [5000, 10000, 20000]),
        "N_bc":         trial.suggest_categorical("N_bc", [100, 200, 400]),
        "N_ic":         trial.suggest_categorical("N_ic", [100, 200, 400]),
        "adam_lr":      trial.suggest_float("adam_lr", 1e-4, 1e-2, log=True),
        "adam_iters":   1500,
        "lbfgs_iters":  500,
    }

    results = train_inverse(config, x_obs_inv, t_obs_inv, u_obs_inv, print_training=False, trial=trial)
    return abs(results["alpha"] - config["true_alpha"])

In [ ]:
import os
if os.path.exists("optuna_inverse.db"):
    os.remove("optuna_inverse.db")


In [ ]:
import joblib
import optuna.visualization as vis

study = optuna.create_study(
    direction="minimize",
    study_name="inverse_pinn_hpo",
    storage="sqlite:///optuna_inverse.db",
    load_if_exists=True,
    pruner=optuna.pruners.MedianPruner(n_startup_trials=15, n_warmup_steps=800, interval_steps=200, n_min_trials=8),
    sampler=optuna.samplers.TPESampler(seed=42)
)

optuna_inverse_search_start = time.time()
study.optimize(
    objective_inverse, 
    n_trials=300,
    show_progress_bar=True
)
optuna_inverse_search_time = time.time() - optuna_inverse_search_start

joblib.dump(study, 'inverse_hpo_study.pkl')

print(f"\nTotal Optuna inverse search time: {optuna_inverse_search_time:.2f}s")
print("\nBest trial:")
print(f"  alpha_error: {study.best_trial.value:.4e}")
print(f"  params: {study.best_trial.params}")

vis.plot_optimization_history(study).show()
vis.plot_param_importances(study).show()

In [ ]:
study = optuna.load_study(
    study_name="inverse_pinn_hpo",
    storage="sqlite:///optuna_inverse.db"
)

print(f"\nBest trial:")
print(f"  Alpha error: {study.best_value:.6e}")
print(f"  Config: {study.best_params}")

# See all top configs
print(f"\nTop 5 trials:")
sorted_trials = sorted(study.trials, key=lambda t: t.value if t.value is not None else float('inf'))
for i, trial in enumerate(sorted_trials[:5], 1):
    print(f"{i}. Alpha error: {trial.value:.6e}")
    print(f"   Params: {trial.params}\n")

# Stats
completed = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
pruned = [t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED]
print(f"Summary:")
print(f"  Completed: {len(completed)}")
print(f"  Pruned: {len(pruned)}")
print(f"  Pruning saved: {len(pruned)/(len(completed)+len(pruned))*100:.1f}% of trials")


In [ ]:
# Confirmation round: retrain each of Optuna's top 3 configs (by search-
# stage alpha error) across 5 FRESH noisy datasets the search stage never
# saw (seeds 1-5, vs. the search stage's seed 0), and pick the winner by
# mean alpha error across those datasets. Since alpha only exists as a
# byproduct of training on some dataset -- there's no way to "query" a
# trained model against new data the way forward's field can be queried at
# new (x,t) points -- the inverse-problem equivalent of forward's held-out
# test grid is a held-out set of simulated noise realizations. This stage
# only decides WHICH CONFIG WINS; the final accuracy number is reported by
# the final-evaluation cell below on yet another, still-untouched set of
# datasets, so no single dataset both picks a winner and reports its
# accuracy.
top_inverse_configs = [
    {**INVERSE_FIXED_CONFIG, **trial.params, "adam_iters": 3000, "lbfgs_iters": 1500}
    for trial in sorted_trials[:3]
]

all_inverse_config_results = []

for i, config in enumerate(top_inverse_configs, 1):
    config_results = []
    for seed in range(1, 6):
        set_seed(seed)
        x_obs_c, t_obs_c, u_obs_c = generate_noisy_data(INVERSE_OBS_CONFIG)
        r = train_inverse(config, x_obs_c, t_obs_c, u_obs_c, print_training=False)
        config_results.append(r)

    errors = [r["alpha_error"] for r in config_results]
    times = [r["train_time"] for r in config_results]
    print(f"\nInverse config {i}:")
    print(f"  Alpha error: {np.mean(errors):.6e} ± {np.std(errors):.6e}")
    print(f"  Time:        {np.mean(times):.2f} ± {np.std(times):.2f} sec")

    all_inverse_config_results.append(config_results)

# Pick the winning config by its MEAN alpha error across the 5 confirmation
# datasets, not by any single individual run's error.
inverse_config_mean_errors = [np.mean([r["alpha_error"] for r in cr]) for cr in all_inverse_config_results]
best_inverse_config_idx = int(np.argmin(inverse_config_mean_errors))
winning_inverse_config = top_inverse_configs[best_inverse_config_idx]

print(f"\nWinning inverse config (by confirmation-round mean alpha error): Config {best_inverse_config_idx + 1}")

In [ ]:
# Final evaluation: retrain the confirmation round's winner -- frozen, no
# further config selection happens here -- across 5 MORE fresh datasets
# (seeds 6-10), never touched by the search stage or the confirmation
# round. This mean +/- std is what should be quoted as the tuned inverse
# PINN's accuracy, and CN-NLS in the next cell is evaluated on these
# identical datasets for a leakage-free final comparison.
final_inverse_results = []
for seed in range(6, 11):
    set_seed(seed)
    x_obs_f, t_obs_f, u_obs_f = generate_noisy_data(INVERSE_OBS_CONFIG)
    r = train_inverse(winning_inverse_config, x_obs_f, t_obs_f, u_obs_f, print_training=False)
    final_inverse_results.append(r)

tuned_inverse_alpha_error_mean = float(np.mean([r["alpha_error"] for r in final_inverse_results]))
tuned_inverse_alpha_error_std = float(np.std([r["alpha_error"] for r in final_inverse_results]))
tuned_inverse_train_time_mean = float(np.mean([r["train_time"] for r in final_inverse_results]))
tuned_inverse_train_time_std = float(np.std([r["train_time"] for r in final_inverse_results]))

print(f"\nFinal tuned inverse PINN (winning config, evaluated on fresh datasets):")
print(f"  Alpha error:   {tuned_inverse_alpha_error_mean:.6e} ± {tuned_inverse_alpha_error_std:.6e}")
print(f"  Training time: {tuned_inverse_train_time_mean:.2f} ± {tuned_inverse_train_time_std:.2f} sec")

# Use whichever seed's alpha error is closest to the mean as a single
# representative run for the illustrative plot below -- mirrors forward's
# representative-model selection in the top-configs cell.
representative_inverse_idx = int(np.argmin(
    [abs(r["alpha_error"] - tuned_inverse_alpha_error_mean) for r in final_inverse_results]))
inverse_results = final_inverse_results[representative_inverse_idx]

In [ ]:
# Illustrative loss-curve / alpha-convergence plot for the winning tuned
# config's representative run (from the retrain loop above) -- mirrors
# forward's representative-run plot, which shows the actual winning model
# rather than a separately hand-set config.
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].semilogy(inverse_results["history_total"], label='Total')
axes[0].semilogy(inverse_results["history_pde"], label='PDE')
axes[0].semilogy(inverse_results["history_bc"], label='BC')
axes[0].semilogy(inverse_results["history_ic"], label='IC')
axes[0].semilogy(inverse_results["history_data"], label='Data')
axes[0].axvline(x=inverse_results["adam_iters"], color='black', linestyle='--', label='Adam → L-BFGS')
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('Loss (log scale)')
axes[0].set_title('Inverse PINN Training Loss')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(inverse_results["history_alpha"], label='Recovered alpha')
axes[1].axhline(y=INVERSE_FIXED_CONFIG["true_alpha"], color='r', linestyle='--', label='True alpha')
axes[1].axvline(x=inverse_results["adam_iters"], color='black', linestyle='--', label='Adam → L-BFGS')
axes[1].set_xlabel('Iteration')
axes[1].set_ylabel('Alpha')
axes[1].set_title('Alpha Convergence')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
true_alpha = INVERSE_FIXED_CONFIG["true_alpha"]

# Run CN-NLS against the SAME 5 final-evaluation datasets as the tuned PINN
# above (each stored on its own result dict by train_inverse), so PINN and
# CN-NLS are compared on identical observations and CN-NLS gets the same
# across-datasets robustness treatment instead of being judged on a single
# noise realization.
cn_nls_results = []
for r in final_inverse_results:
    best_alpha, mse_history, alpha_candidates, cn_nls_time = cn_nls_baseline(
        {"n_alpha_candidates": 200}, r["x_obs"], r["t_obs"], r["u_obs"])
    cn_nls_results.append({
        "best_alpha": best_alpha,
        "mse_history": mse_history,
        "alpha_candidates": alpha_candidates,
        "cn_nls_time": cn_nls_time,
        "alpha_error": abs(best_alpha - true_alpha),
    })

cn_nls_alpha_errors = [c["alpha_error"] for c in cn_nls_results]
cn_nls_times = [c["cn_nls_time"] for c in cn_nls_results]
cn_nls_alpha_error_mean = float(np.mean(cn_nls_alpha_errors))
cn_nls_alpha_error_std = float(np.std(cn_nls_alpha_errors))
cn_nls_time_mean = float(np.mean(cn_nls_times))
cn_nls_time_std = float(np.std(cn_nls_times))

# Use whichever draw's alpha error is closest to the mean as a single
# representative run for the landscape plot below -- illustrative only.
representative_cn_idx = int(np.argmin([abs(c["alpha_error"] - cn_nls_alpha_error_mean) for c in cn_nls_results]))
best_alpha = cn_nls_results[representative_cn_idx]["best_alpha"]
mse_history = cn_nls_results[representative_cn_idx]["mse_history"]
alpha_candidates = cn_nls_results[representative_cn_idx]["alpha_candidates"]
cn_nls_time = cn_nls_results[representative_cn_idx]["cn_nls_time"]
cn_nls_alpha_error = cn_nls_results[representative_cn_idx]["alpha_error"]

print(f"CN-NLS recovered alpha: {best_alpha:.6f} | True alpha: {true_alpha} | Error: {cn_nls_alpha_error:.6f}")

plt.figure(figsize=(8, 4))
plt.plot(alpha_candidates, mse_history, label='MSE')
plt.axvline(x=best_alpha, color='b', linestyle='--', label=f'Best alpha ({best_alpha:.4f})')
plt.axvline(x=true_alpha, color='r', linestyle='--', label=f'True alpha ({true_alpha})')
plt.xlabel('Alpha candidate')
plt.ylabel('MSE')
plt.title('CN-NLS Objective Landscape (representative draw)')
plt.legend()
plt.grid(True)
plt.show()

# Baseline / Tuned / CN-NLS comparison table, mirroring forward's final
# summary table -- alpha error and cost, side by side. Row label says
# "runs" rather than "seeds": Baseline's 5 runs share one dataset (only
# training seed varies), but Tuned's 5 runs each use a different dataset,
# and CN-NLS has no training randomness at all -- its variation is entirely
# from the different datasets. "Seeds" would only be accurate for Baseline.
print(f"\n{'':25} {'Baseline':>14} {'Tuned':>14} {'CN-NLS':>14}")
print(f"{'-'*70}")
print(f"{'Alpha error':25} {'N/A':>14} {'N/A':>14} {cn_nls_alpha_error:>14.4e}")
print(f"{'  mean ± std (5 runs)':25} {f'{np.mean(baseline_alpha_errors):.4e}±{np.std(baseline_alpha_errors):.4e}':>14} {f'{tuned_inverse_alpha_error_mean:.4e}±{tuned_inverse_alpha_error_std:.4e}':>14} {f'{cn_nls_alpha_error_mean:.4e}±{cn_nls_alpha_error_std:.4e}':>14}")
print(f"{'Training/opt time (s)':25} {'N/A':>14} {'N/A':>14} {cn_nls_time:>14.2f}")
print(f"{'  mean ± std (5 runs)':25} {f'{np.mean(baseline_inverse_times):.2f}±{np.std(baseline_inverse_times):.2f}':>14} {f'{tuned_inverse_train_time_mean:.2f}±{tuned_inverse_train_time_std:.2f}':>14} {f'{cn_nls_time_mean:.2f}±{cn_nls_time_std:.2f}':>14}")
print(f"{'Optuna search time (s)':25} {'N/A':>14} {optuna_inverse_search_time:>14.2f} {'N/A':>14}")

In [ ]:
# Sensitivity sweep 1: vary observation noise (noise_std) at fixed
# N_obs=50, comparing the tuned PINN (winning_inverse_config, frozen from
# the confirmation round above -- NOT re-tuned per point, since the point
# is testing how robust the already-selected config is to noise, not
# re-running the expensive HPO search 4 more times) against CN-NLS at each
# noise level. The noise_std=0.05 point reuses the already-computed final
# results above instead of retraining a duplicate of the identical
# experiment. Seeds 11-15 used here -- search=0, confirmation=1-5,
# final-eval=6-10, this sweep=11-15, the N_obs sweep below=16-20 -- so no
# dataset is reused across any two stages, including between the sweeps.
NOISE_SWEEP_VALUES = [0.01, 0.05, 0.1, 0.2]
noise_sweep_results = {}
noise_sweep_start = time.time()

for noise_std in NOISE_SWEEP_VALUES:
    if noise_std == INVERSE_OBS_CONFIG["noise_std"]:
        noise_sweep_results[noise_std] = {
            "pinn_alpha_error_mean": tuned_inverse_alpha_error_mean,
            "pinn_alpha_error_std": tuned_inverse_alpha_error_std,
            "cn_nls_alpha_error_mean": cn_nls_alpha_error_mean,
            "cn_nls_alpha_error_std": cn_nls_alpha_error_std,
        }
        print(f"noise_std={noise_std}: reused final-evaluation results (identical config)")
        continue

    sweep_obs_config = {**INVERSE_OBS_CONFIG, "noise_std": noise_std}
    pinn_errors, cn_errors = [], []
    for seed in range(11, 16):
        set_seed(seed)
        x_obs_s, t_obs_s, u_obs_s = generate_noisy_data(sweep_obs_config)
        r = train_inverse(winning_inverse_config, x_obs_s, t_obs_s, u_obs_s, print_training=False)
        pinn_errors.append(r["alpha_error"])

        best_alpha_s, _, _, _ = cn_nls_baseline({"n_alpha_candidates": 200}, r["x_obs"], r["t_obs"], r["u_obs"])
        cn_errors.append(abs(best_alpha_s - true_alpha))

    noise_sweep_results[noise_std] = {
        "pinn_alpha_error_mean": float(np.mean(pinn_errors)),
        "pinn_alpha_error_std": float(np.std(pinn_errors)),
        "cn_nls_alpha_error_mean": float(np.mean(cn_errors)),
        "cn_nls_alpha_error_std": float(np.std(cn_errors)),
    }
    print(f"noise_std={noise_std}: PINN alpha error {np.mean(pinn_errors):.4e} ± {np.std(pinn_errors):.4e} | "
          f"CN-NLS alpha error {np.mean(cn_errors):.4e} ± {np.std(cn_errors):.4e}")

noise_sweep_time = time.time() - noise_sweep_start
print(f"\nNoise sensitivity sweep total time: {noise_sweep_time:.2f}s")

plt.figure(figsize=(7, 5))
xs = sorted(noise_sweep_results.keys())
pinn_means = [noise_sweep_results[x]["pinn_alpha_error_mean"] for x in xs]
pinn_stds = [noise_sweep_results[x]["pinn_alpha_error_std"] for x in xs]
cn_means = [noise_sweep_results[x]["cn_nls_alpha_error_mean"] for x in xs]
cn_stds = [noise_sweep_results[x]["cn_nls_alpha_error_std"] for x in xs]
plt.errorbar(xs, pinn_means, yerr=pinn_stds, marker='o', label='Tuned PINN', capsize=4)
plt.errorbar(xs, cn_means, yerr=cn_stds, marker='s', label='CN-NLS', capsize=4)
plt.xlabel('Observation noise (noise_std)')
plt.ylabel('Alpha error')
plt.title(f'Alpha Error vs. Observation Noise (N_obs={INVERSE_OBS_CONFIG["N_obs"]})')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Sensitivity sweep 2: vary observation count (N_obs) at fixed
# noise_std=0.05, same methodology as the noise sweep above -- frozen
# winning_inverse_config, N_obs=50 point reuses the already-computed final
# results, seeds 16-20 (distinct from every other stage including the
# noise sweep's 11-15).
NOBS_SWEEP_VALUES = [10, 25, 50, 100]
nobs_sweep_results = {}
nobs_sweep_start = time.time()

for n_obs in NOBS_SWEEP_VALUES:
    if n_obs == INVERSE_OBS_CONFIG["N_obs"]:
        nobs_sweep_results[n_obs] = {
            "pinn_alpha_error_mean": tuned_inverse_alpha_error_mean,
            "pinn_alpha_error_std": tuned_inverse_alpha_error_std,
            "cn_nls_alpha_error_mean": cn_nls_alpha_error_mean,
            "cn_nls_alpha_error_std": cn_nls_alpha_error_std,
        }
        print(f"N_obs={n_obs}: reused final-evaluation results (identical config)")
        continue

    sweep_obs_config = {**INVERSE_OBS_CONFIG, "N_obs": n_obs}
    pinn_errors, cn_errors = [], []
    for seed in range(16, 21):
        set_seed(seed)
        x_obs_s, t_obs_s, u_obs_s = generate_noisy_data(sweep_obs_config)
        r = train_inverse(winning_inverse_config, x_obs_s, t_obs_s, u_obs_s, print_training=False)
        pinn_errors.append(r["alpha_error"])

        best_alpha_s, _, _, _ = cn_nls_baseline({"n_alpha_candidates": 200}, r["x_obs"], r["t_obs"], r["u_obs"])
        cn_errors.append(abs(best_alpha_s - true_alpha))

    nobs_sweep_results[n_obs] = {
        "pinn_alpha_error_mean": float(np.mean(pinn_errors)),
        "pinn_alpha_error_std": float(np.std(pinn_errors)),
        "cn_nls_alpha_error_mean": float(np.mean(cn_errors)),
        "cn_nls_alpha_error_std": float(np.std(cn_errors)),
    }
    print(f"N_obs={n_obs}: PINN alpha error {np.mean(pinn_errors):.4e} ± {np.std(pinn_errors):.4e} | "
          f"CN-NLS alpha error {np.mean(cn_errors):.4e} ± {np.std(cn_errors):.4e}")

nobs_sweep_time = time.time() - nobs_sweep_start
print(f"\nN_obs sensitivity sweep total time: {nobs_sweep_time:.2f}s")

plt.figure(figsize=(7, 5))
xs = sorted(nobs_sweep_results.keys())
pinn_means = [nobs_sweep_results[x]["pinn_alpha_error_mean"] for x in xs]
pinn_stds = [nobs_sweep_results[x]["pinn_alpha_error_std"] for x in xs]
cn_means = [nobs_sweep_results[x]["cn_nls_alpha_error_mean"] for x in xs]
cn_stds = [nobs_sweep_results[x]["cn_nls_alpha_error_std"] for x in xs]
plt.errorbar(xs, pinn_means, yerr=pinn_stds, marker='o', label='Tuned PINN', capsize=4)
plt.errorbar(xs, cn_means, yerr=cn_stds, marker='s', label='CN-NLS', capsize=4)
plt.xlabel('Number of observations (N_obs)')
plt.ylabel('Alpha error')
plt.title(f'Alpha Error vs. Observation Count (noise_std={INVERSE_OBS_CONFIG["noise_std"]})')
plt.legend()
plt.grid(True)
plt.show()